# t-SNE (t-distributed Stochastic Neighbor Embedding) / UMAP (Uniform Manifold Approximation and Projection)

_Classical ML_

**Squash high-dimensional data to 2-D while keeping neighbors together.**

t-SNE and UMAP are tools for seeing high-dimensional data on a flat screen.

     They place each point in 2-D so that points close in the original space stay close on the map.

---

This notebook is a practice scaffold for the **t-SNE (t-distributed Stochastic Neighbor Embedding) / UMAP (Uniform Manifold Approximation and Projection)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns** — each sample is an 8x8 grid of pixel intensities (0–16).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("image array:", digits.images.shape, " labels:", digits.target.shape)
samples = list(zip(digits.images, digits.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — scikit-learn

We'll project a synthetic 10-dimensional dataset of three clusters down to 2-D with t-SNE, then *measure* how well the clusters stay apart. We build it in three steps: (1) make the data and embed it, (2) find each cluster's centre in the map, (3) score the separation.

### Step 1 — Make 10-D blobs and embed them in 2-D

We generate 150 points spread across three clusters in 10-dimensional space. t-SNE then squashes them to 2-D, placing each point so that near neighbours in the original space stay near on the map. `init="pca"` and a fixed `random_state` make the layout reproducible.

In [ ]:
from sklearn.datasets import make_blobs
from sklearn.manifold import TSNE

# 150 points, 10 features each, grouped into 3 clusters.
X, labels = make_blobs(n_samples=150, n_features=10, centers=3,
                       cluster_std=1.0, random_state=0)

# Project the 10-D points down to a 2-D map.
emb = TSNE(n_components=2, perplexity=30, init="pca",
           random_state=0).fit_transform(X)
print("embedded shape:", emb.shape)

### Step 2 — Find each cluster's centre in the 2-D map

For every cluster `k`, we average the 2-D positions of its points to get a **centroid** — the cluster's centre on the map. These centroids let us quantify how tight each cluster is and how far apart the clusters sit.

In [ ]:
# Centroid (mean 2-D position) of each of the 3 clusters.
cents = np.array([emb[labels == k].mean(axis=0) for k in range(3)])
print("cluster centroids:\n", np.round(cents, 2))

### Step 3 — Score the separation

The **within-cluster radius** is the average distance from points to their own centroid (smaller = tighter clusters). The **between-cluster gap** is the average distance between centroids (larger = clusters further apart). Their ratio is a single separation score: a high ratio means t-SNE kept the three groups cleanly apart.

In [ ]:
# Average distance from each point to its own cluster centroid.
within = np.mean([np.linalg.norm(emb[labels == k] - cents[k], axis=1).mean()
                  for k in range(3)])

# Average distance between every pair of cluster centroids.
between = np.mean([np.linalg.norm(cents[i] - cents[j])
                   for i in range(3) for j in range(i + 1, 3)])

print("avg within-cluster radius :", round(within, 2))
print("avg between-cluster gap   :", round(between, 2))
print("separation ratio          :", round(between / within, 2))

## Visualize the data & results

_Does t-SNE expose the digit structure in real handwritten images?_

### Step 1 — Embed real handwritten digits with t-SNE

Now the same idea on real data. Each handwritten digit is an 8x8 image, i.e. a 64-dimensional point. We keep digits 0–4 and let t-SNE project them to 2-D, hoping each digit forms its own cloud.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.manifold import TSNE

digits = load_digits()                   # 8x8 handwritten-digit images

# Keep only digits 0,1,2,3,4 to keep the plot readable.
keep = np.isin(digits.target, [0, 1, 2, 3, 4])
X, y = digits.data[keep], digits.target[keep]

# Project the 64-D images down to a 2-D map.
emb = TSNE(n_components=2, perplexity=30, init="pca",
           random_state=0).fit_transform(X)
print("embedded shape:", emb.shape)

### Step 2 — Plot the 2-D map, coloured by digit

We draw each digit's points in its own colour. If t-SNE captured the structure, the five colours form five separate clusters — visual proof that the high-dimensional digit images really do live on well-separated regions.

In [ ]:
colors = ["#4ea1ff", "#7ee787", "#ffb454", "#c89bff", "#ff7b72"]

# Scatter each digit's points in its own colour.
for d in range(5):
    pts = emb[y == d]
    plt.scatter(pts[:, 0], pts[:, 1], c=colors[d], s=12, label="digit %d" % d)

plt.title("t-SNE of handwritten digits (8x8 images) projected to 2-D")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.legend()
plt.show()